In [1]:
from datasets import load_dataset

dolly = load_dataset("databricks/databricks-dolly-15k")
dolly_df = dolly["train"].to_pandas()

print(dolly_df["category"].value_counts())

category
open_qa                   3742
general_qa                2191
classification            2136
closed_qa                 1773
brainstorming             1766
information_extraction    1506
summarization             1188
creative_writing           709
Name: count, dtype: int64


In [2]:
import pandas as pd

categories = dolly_df["category"].unique()
n_par_cat = 1565 // len(categories)   # ~195 par catégorie

echantillons = []
for cat in categories:
    sous_ensemble = dolly_df[dolly_df["category"] == cat]
    n = min(n_par_cat, len(sous_ensemble))   # sécurité si une catégorie est petite
    echantillons.append(sous_ensemble.sample(n=n, random_state=42))

benins_equilibres = pd.concat(echantillons, ignore_index=True)
benins_equilibres = benins_equilibres[["instruction"]].rename(columns={"instruction": "text"})
benins_equilibres["label"] = 0

print("Bénins prélevés :", len(benins_equilibres))
print("\nRépartition par catégorie d'origine (vérification) :")
# petit contrôle : on recompte via un merge rapide

Bénins prélevés : 1560

Répartition par catégorie d'origine (vérification) :


In [3]:
import pandas as pd

ancien = pd.read_csv("../data/dataset.csv")
attaques = ancien[ancien["label"] == 1].copy()
print("Attaques récupérées :", len(attaques))

dataset_v2 = pd.concat([attaques, benins_equilibres], ignore_index=True)
dataset_v2 = dataset_v2.drop_duplicates(subset=["text"]).reset_index(drop=True)
dataset_v2 = dataset_v2.sample(frac=1, random_state=42).reset_index(drop=True)

print("\nTaille dataset v2 :", len(dataset_v2))
print(dataset_v2["label"].value_counts())
print(dataset_v2["label"].value_counts(normalize=True).round(3))

Attaques récupérées : 1567

Taille dataset v2 : 3120
label
1    1567
0    1553
Name: count, dtype: int64
label
1    0.502
0    0.498
Name: proportion, dtype: float64


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sentence_transformers import SentenceTransformer

# Split
X_train, X_test, y_train, y_test = train_test_split(
    dataset_v2["text"], dataset_v2["label"],
    test_size=0.2, random_state=42, stratify=dataset_v2["label"]
)

# Embeddings
model = SentenceTransformer("all-MiniLM-L6-v2")
emb_train = model.encode(X_train.tolist(), show_progress_bar=True)
emb_test = model.encode(X_test.tolist(), show_progress_bar=True)

# Entraînement
clf = LogisticRegression(max_iter=1000)
clf.fit(emb_train, y_train)

# Évaluation
y_pred = clf.predict(emb_test)
print(classification_report(y_test, y_pred, target_names=["bénin", "malveillant"]))
print("Matrice de confusion :")
print(confusion_matrix(y_test, y_pred))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/78 [00:00<?, ?it/s]

Batches:   0%|          | 0/20 [00:00<?, ?it/s]

              precision    recall  f1-score   support

       bénin       0.96      0.98      0.97       311
 malveillant       0.98      0.96      0.97       313

    accuracy                           0.97       624
   macro avg       0.97      0.97      0.97       624
weighted avg       0.97      0.97      0.97       624

Matrice de confusion :
[[306   5]
 [ 14 299]]


In [6]:
benins_varies = [
    "Can you help me write a cover letter?",
    "Please write a short poem about the sea.",
    "Write a summary of this article for me.",
    "Could you explain how photosynthesis works?",
    "Help me draft an email to my professor.",
    "Tell me a story about a dragon.",
    "Give me some tips to improve my resume.",
    "Explain the rules of chess to a beginner.",
]
emb = model.encode(benins_varies)
preds = clf.predict(emb)
for texte, p in zip(benins_varies, preds):
    label = "FAUX POSITIF" if p == 1 else "bénin"
    print(f"{label}  |  {texte}")

bénin  |  Can you help me write a cover letter?
bénin  |  Please write a short poem about the sea.
bénin  |  Write a summary of this article for me.
bénin  |  Could you explain how photosynthesis works?
bénin  |  Help me draft an email to my professor.
bénin  |  Tell me a story about a dragon.
bénin  |  Give me some tips to improve my resume.
bénin  |  Explain the rules of chess to a beginner.


In [7]:
import joblib, os
os.makedirs("../models", exist_ok=True)
joblib.dump(clf, "../models/baseline_logreg_v2.joblib")
print("Modèle v2 sauvegardé.")

Modèle v2 sauvegardé.


In [8]:
dataset_v2.to_csv("../data/dataset_v2.csv", index=False)

# Phase 2 — Baseline et correction du biais

# Baseline (v1)
- Embeddings all-MiniLM-L6-v2 + régression logistique → F1 = 0.95.
- Score élevé mais trompeur : test manuel révélant des faux positifs sur des
  demandes de rédaction légitimes (« write a cover letter »).

# Diagnostic
- Biais de source/registre : les bénins (Dolly) étaient majoritairement des
  questions factuelles (open_qa). Le modèle avait appris à reconnaître le registre,
  pas la malveillance (shortcut learning).

# Correction (v2)
- Rééchantillonnage équilibré des 8 catégories Dolly pour couvrir tous les registres.
- Résultat : faux positifs éliminés sur les cas de test, F1 = 0.97,
  faux négatifs divisés par ~2 (27 → 14).

# Leçon
Quand un modèle échoue, examiner la donnée avant l'algorithme.